# Description

In this notebook, we benchmark SINDy algorithm on the Korns benchmarks.

In [1]:
from config.korns_config import BENCH, SINDY, FEATURE_NAMES
from src.korns_core import RunConfig, load_korns_hdf5, run_benchmark, SRFitResult

from dataclasses import dataclass
from typing import List, Dict, Tuple, Callable
import numpy as np
import sympy as sp

from pysindy.feature_library import PolynomialLibrary, CustomLibrary, ConcatLibrary
from pysindy.optimizers import STLSQ


def to_sympy(feature_name: str, symbols: Dict[str, sp.Symbol]) -> sp.Expr:
    return sp.sympify(feature_name.replace("^", "**").replace(" ", "*"), locals=symbols)


def make_custom_library(cfg) -> CustomLibrary:
    clip_exp = float(cfg.clip_exp)
    log_eps = float(cfg.log_eps)
    div_eps = float(cfg.div_eps)
    sqrt_abs = bool(cfg.sqrt_abs)

    def sin_(x):  return np.nan_to_num(np.sin(x),  nan=0.0, posinf=0.0, neginf=0.0)
    def cos_(x):  return np.nan_to_num(np.cos(x),  nan=0.0, posinf=0.0, neginf=0.0)
    def tan_(x):  return np.nan_to_num(np.tan(x),  nan=0.0, posinf=0.0, neginf=0.0)
    def tanh_(x): return np.nan_to_num(np.tanh(x), nan=0.0, posinf=0.0, neginf=0.0)

    def exp_(x):
        y = np.exp(np.clip(x, -clip_exp, clip_exp))
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    def log_(x):
        y = np.log(np.abs(x) + log_eps)
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    def sqrt_(x):
        y = np.sqrt(np.abs(x)) if sqrt_abs else np.sqrt(x)
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    def div_(a, b):
        y = a / (b + div_eps)
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    unary: Dict[str, Tuple[Callable, Callable]] = {
        "sin":  (sin_,  lambda x: f"sin({x})"),
        "cos":  (cos_,  lambda x: f"cos({x})"),
        "tan":  (tan_,  lambda x: f"tan({x})"),
        "tanh": (tanh_, lambda x: f"tanh({x})"),
        "exp":  (exp_,  lambda x: f"exp({x})"),
        "log":  (log_,  lambda x: f"log({x})"),
        "sqrt": (sqrt_, lambda x: f"sqrt({x})"),
    }

    funcs: List[Callable] = []
    names: List[Callable] = []

    for op in cfg.unary_ops:
        f, n = unary[op]
        funcs.append(f)
        names.append(n)

    if "/" in cfg.binary_ops:
        funcs.append(div_)
        names.append(lambda a, b: f"({a})/({b})")

    return CustomLibrary(library_functions=funcs, function_names=names, include_bias=False)


@dataclass
class SINDyKornsRegressor:
    name: str = "sindy"
    cfg: object = SINDY

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        X_train = np.asarray(X_train, dtype=np.float64)
        X_test = np.asarray(X_test, dtype=np.float64)
        y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)

        input_names = list(FEATURE_NAMES[: X_train.shape[1]])

        poly = PolynomialLibrary(
            degree=int(self.cfg.poly_degree),
            include_interaction=bool(self.cfg.include_interaction),
            include_bias=bool(self.cfg.include_bias),
        )
        custom = make_custom_library(self.cfg)
        library = ConcatLibrary([poly, custom])

        library.fit(X_train)

        theta_train = np.asarray(library.transform(X_train), dtype=np.float64)
        theta_test = np.asarray(library.transform(X_test), dtype=np.float64)

        theta_train = np.nan_to_num(theta_train, nan=0.0, posinf=0.0, neginf=0.0)
        theta_test = np.nan_to_num(theta_test, nan=0.0, posinf=0.0, neginf=0.0)

        m = float(self.cfg.max_feature_abs)
        theta_train = np.clip(theta_train, -m, m)
        theta_test = np.clip(theta_test, -m, m)

        feature_names = library.get_feature_names(input_features=input_names)

        k = int(getattr(self.cfg, "max_library_features", 0) or 0)
        if k > 0 and theta_train.shape[1] > k:
            theta_train = theta_train[:, :k]
            theta_test = theta_test[:, :k]
            feature_names = feature_names[:k]

        if bool(self.cfg.drop_nonfinite_rows):
            keep = np.isfinite(theta_train).all(axis=1) & np.isfinite(y_train).all(axis=1)
            theta_train = theta_train[keep]
            y_train = y_train[keep]

        optimizer = STLSQ(
            threshold=float(self.cfg.threshold),
            alpha=float(self.cfg.alpha),
            max_iter=int(self.cfg.max_iter),
            normalize_columns=bool(self.cfg.normalize_columns),
            verbose=bool(self.cfg.stlsq_verbose),
        )
        optimizer.fit(theta_train, y_train)

        coef = np.asarray(optimizer.coef_, dtype=np.float64)
        coef = coef[0] if coef.ndim == 2 else coef
        intercept = float(np.atleast_1d(getattr(optimizer, "intercept_", 0.0))[0])

        coef = np.nan_to_num(coef, nan=0.0, posinf=0.0, neginf=0.0)
        intercept = float(np.nan_to_num(intercept, nan=0.0, posinf=0.0, neginf=0.0))

        y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
        y_pred_test = (theta_test @ coef.reshape(-1, 1)).reshape(-1) + intercept

        expr = None
        try:
            symbols = {n: sp.Symbol(n) for n in input_names}
            tol = float(self.cfg.coef_zero_tol)

            terms = []
            for c, fname in zip(coef.tolist(), feature_names):
                if abs(c) <= tol:
                    continue
                terms.append(sp.Float(c) * to_sympy(fname, symbols))

            expr = sp.Add(*terms) if terms else sp.Float(0.0)
            if abs(intercept) > tol:
                expr = expr + sp.Float(intercept)
        except Exception:
            expr = None

        return SRFitResult(
            expr=expr,
            y_pred_train=np.asarray(y_pred_train, dtype=np.float64).reshape(-1),
            y_pred_test=np.asarray(y_pred_test, dtype=np.float64).reshape(-1),
            metadata=None,
        )


run_cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

datasets = load_korns_hdf5(run_cfg.hdf5_path)

rows = run_benchmark(
    datasets=datasets,
    algorithms=[SINDyKornsRegressor()],
    config=run_cfg,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path=SINDY.results_csv_path,
)


[PROBLEM] P1
[GT] 24.3*x3 + 1.57
[ALGO] sindy
[RUN START] run_id=0 seed=14176308741000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * |w|_2
         0 ... 4.1005e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
         1 ... 3.2854e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
[PRED] 24.3*x3 + 1.56999999999999
[PROBLEM] P2
[GT] 0.23 + 4.73333333333333*(x1 + x3)/x4
[ALGO] sindy
[RUN START] run_id=0 seed=14176308742000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * |w|_2
         0 ... 2.5271e+05 ... 4.7225e-02 ...         30 ... 2.5271e+05


/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: divide by zero encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: overflow encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_716

[PRED] 0.0548925429296839*x0**2 + 0.0288189938996802*x0*x1 - 0.0329282094129827*x0*x2 + 0.0161170500327827*x0*x3 + 0.0304846349670816*x0*x4 - 0.00210857955085164*x0 + 0.000531574084885562*x1**2 - 0.0965048660253908*x1*x2 + 0.101424885464241*x1*x3 + 0.519699621094771*x1*x4 - 0.163983451979242*x1 - 0.0136242147106641*x2**2 - 0.0348106817888895*x2*x3 + 0.020029880943461*x2*x4 + 0.11452319665054*x2 + 0.0625029848843601*x3**2 + 0.531180065739177*x3*x4 - 0.402331772186595*x3 + 0.0126489015973401*x4**2 + 0.0580823132226693*x4 - 0.885981246600113*sin(x0) - 1.0605741498325*sin(x1) + 1.01398086235589*sin(x2) + 0.062367555886616*sin(x3) + 0.538931792636787*sin(x4) - 0.311826829438993*cos(x0) - 0.0491631831074237*cos(x1) - 0.53976093595917*cos(x2) - 0.14690938865178*cos(x3) - 0.783358699418401
[PROBLEM] P3
[GT] -5.41 + 1.63333333333333*(-x0 + x1/x4 + x3)/x4
[ALGO] sindy
[RUN START] run_id=0 seed=14176308743000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a 

/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: divide by zero encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: overflow encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_716

[PRED] 0.0783120013745499*x0**2 - 0.0152295896400314*x0*x1 + 0.104508576910564*x0*x2 - 0.0813837099438342*x0*x3 - 0.184426663421464*x0*x4 + 0.0353259186983992*x0 + 0.0141210858070121*x1**2 - 0.0352385897067146*x1*x2 + 0.0332466799992479*x1*x3 - 0.0289295941645284*x1*x4 + 1.31314014174469*x1 + 0.0657837053576209*x2**2 + 0.047056917038094*x2*x3 - 0.0176765591911644*x2*x4 + 0.0506704424698602*x2 + 0.0445955344319238*x3**2 + 0.179722613812964*x3*x4 - 0.0778540901017104*x3 - 0.0107296340396528*x4**2 - 0.0266200028352992*x4 - 0.248808463918416*sin(x0) + 1.48941331034629*sin(x1) + 0.924530226587548*sin(x2) - 0.101256526430028*sin(x3) - 0.036887161247618*sin(x4) + 1.00674911975418*cos(x0) - 0.521745059520649*cos(x1) - 0.280796782703642*cos(x2) + 0.253054342886661*cos(x3) - 6.75367761132818
[PROBLEM] P4
[GT] 0.13*sin(x2) - 2.3
[ALGO] sindy
[RUN START] run_id=0 seed=14176308744000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * |w|_2
         0 ... 3.334

...

[PRED] -0.25567799406079*x0**2 + 0.0605313979424046*x0*x1 + 0.125775597235196*x0*x2 + 0.0867863153724611*x0*x3 - 0.0101218049035188*x0*x4 + 0.0734111124893288*x0 + 0.117843951102918*x1**2 - 0.164394313226962*x1*x2 - 0.106327963309696*x1*x3 - 0.0269331537086771*x1*x4 + 0.354674625701424*x1 - 0.0128628791605825*x2**2 - 0.053195417366808*x2*x3 + 0.046507369725317*x2*x4 - 0.0960599367262511*x2 + 0.0752081491728354*x3**2 - 0.0676253164057305*x3*x4 + 0.0597820243316766*x3 - 0.0796394078754599*x4**2 - 0.211830908828163*x4 + 1.51068696266825*sin(x0) + 0.0667089784634984*sin(x1) + 1.08155615299094*sin(x2) - 0.621899716217004*sin(x3) + 0.604577908352189*sin(x4) - 1.69481303639072*cos(x0) + 1.70272833509187*cos(x1) - 0.523097614666598*cos(x2) + 1.41409362386703*cos(x3) + 30.4865974136544
[PROBLEM] P14
[GT] -4.2*(cos(x0) - tan(x1))*tanh(x2)/sin(x3) + 22.0
[ALGO] sindy
[RUN START] run_id=0 seed=14176308754000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * 

/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: divide by zero encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: overflow encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_716

[PRED] -0.0171969932009128*x0**2 - 0.118662966434609*x0*x1 + 0.0718049049598844*x0*x2 - 0.103811120071258*x0*x3 - 0.0147643135108814*x0*x4 - 0.17864806611019*x0 - 0.159303676842583*x1**2 + 0.00721934543987861*x1*x2 + 0.0994243756034942*x1*x3 - 0.0166761616000244*x1*x4 + 0.54050866452469*x1 + 0.085052666755952*x2**2 + 0.0317345147973631*x2*x3 + 0.128335853280815*x2*x4 - 0.574119601252107*x2 - 0.0871653962661568*x3**2 + 0.0739210563303089*x3*x4 - 0.443785733034921*x3 + 0.132676785418581*x4**2 + 0.439919031824516*x4 + 0.399170502631176*sin(x0) + 0.194981302679786*sin(x1) + 0.464017956994279*sin(x2) + 0.350361994541446*sin(x3) + 0.903176670354901*sin(x4) + 0.339245097610756*cos(x0) - 2.07229947252275*cos(x1) - 0.184567578802361*cos(x2) - 2.22797235079551*cos(x3) + 19.7570586790591
[PROBLEM] P15
[GT] -6.0*(log(x2) - tan(x3))*exp(-x1)*tan(x0) + 12.0
[ALGO] sindy
[RUN START] run_id=0 seed=14176308755000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * 

/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: divide by zero encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_7167/4025804205.py:130: RuntimeWarning: overflow encountered in matmul
  y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
/var/folders/41/v78ybjmx7vx3_5kr7kvcbk84h7cx3s/T/ipykernel_716

[PRED] 0.0552040624342987*x0**2 + 0.0434960884903999*x0*x1 + 0.0999395116779839*x0*x2 - 0.0719340115759782*x0*x3 + 0.0162906562267735*x0*x4 - 0.465649557831355*x0 + 0.0285767592186915*x1**2 - 0.189627198743977*x1*x2 - 0.0161420328856301*x1*x3 - 0.0468288418694582*x1*x4 + 0.441535775270012*x1 + 3.09586954983417*x2**2 - 0.0702425442283876*x2*x3 + 0.0890356010044307*x2*x4 - 14.6297920207021*x2 - 0.0694198287052675*x3**2 - 0.118302550561786*x3*x4 + 0.182035261676628*x3 + 0.09323528117484*x4**2 - 0.21057449020913*x4 + 0.334226098213635*sin(x0) - 0.670882939013123*sin(x1) + 4.78419116535255*sin(x2) + 2.63996257384836*sin(x3) + 1.54295275125281*sin(x4) - 1.09794726417766*cos(x0) + 1.01837530890343*cos(x1) - 6.47090913143419*cos(x2) - 0.790481187368429*cos(x3) + 18.5283414456535
